# Lab 8: Gold-Standard Construction, NIST ASR Evaluation & Bootstrap Confidence Intervals
**Course:** CIS 531/731: Programming Techniques for Data Science and Analytics (Fall 2026)  
**Date:** Friday, November 6, 2026  
**Phase 5:** Evaluate & Deploy | **Module 9:** Evaluating Models: Inferential Statistics & ASR Evaluation  
**Culminates In:** Problem Set 8 (PS 8), Machine Problem 9 (MP 9), and HW 12 Final Term-Project Evaluation

---

## Lab Objectives
In Lectures 28 and 29, we established two foundational laws of defensible data science:
1. **A point estimate without a confidence interval is mathematically invalid** (Lecture 28). Every reported Macro-F1 score, Word Error Rate (WER), or generational drift trajectory must be accompanied by a **95% Non-Parametric Percentile Bootstrap Confidence Interval** (B = 1000).
2. **An evaluation is only as valid as the ground truth it is measured against** (Lectures 29 & 30). Using an LLM (e.g., GPT/Claude) or an ASR model (e.g., Whisper Large) to generate your reference test set introduces **methodological circularity**--measuring model agreement rather than true accuracy.

In this execution lab, you will:
1. **Enforce Course Invariant INV-17 & The Annotation Cap:** Draw a stratified sample capped at **N <= 50 objects (or <= 2 hours of human annotation labor)** and log human-verified ground truth (source='gold').
2. **Implement a NIST OpenASR Transcript Normalization Pipeline:** Standardize casing, punctuation, digit-to-word expansion, and disfluencies (<hesitation>) so acoustic accuracy is never confounded with formatting.
3. **Compute Exact Levenshtein Alignment, Word Error Rate (WER), & Entity Error Rate (EER):** Extract exact Substitution (S), Deletion (D), Insertion (I), and Correct (C) counts, and measure domain-specific Named Entity preservation.
4. **Bridge ASR to Downstream Trope Classification (Lab 6 Integration):** Evaluate a Baseline vs. Alternative classifier on your human gold set using scikit-learn.
5. **Construct 95% Percentile Bootstrap Confidence Intervals (B = 1000):** Quantify uncertainty on WER, Macro-F1, paired model differences (delta_F1 = F1_alt - F1_base), and stratified generational cohort drift.

## Part 1: Environment Setup, Stratified Sampling & The Gold-Standard Schema (INV-17)

### Architectural Rules for Gold-Standard Construction
* **INV-17 (Human Ground-Truth Mandate):** Every transcript reference (`gold_transcript`) and thematic label (`gold_trope`) in your gold evaluation split **must** be authored by a human student (`source='gold'`). Never use LLM pseudo-labels as ground truth in your evaluation split.
* **Thermal Load & Annotation Cap:** Individual annotation for this lab and MP 9 is strictly capped at **N <= 50 items or <= 2 hours of human effort**. For 2-3 person teams, pooled gold sets in HW 12 must contain **50-200 items**.
* **Stratified Sampling Across Cohorts/Conditions:** Never annotate only the first 5 minutes of a broadcast or a single decade. Sample across temporal cohorts (`1990s`-`2020s`) or acoustic conditions (`clean_studio`, `noisy_caller`, `music_bed`) so your 50-item gold set captures the true variance of the corpus.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import re
import time
import json
import hashlib
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, classification_report

np.random.seed(42)
print('=== CIS 531/731 Lab 8: Gold-Standard Construction & Evaluation Environment Ready ===')

=== CIS 531/731 Lab 8: Gold-Standard Construction & Evaluation Environment Ready ===


In [5]:
import json
import pandas as pd
import numpy as np

import json
import pandas as pd
import numpy as np

all_chunks = []
with open("../silver_vocadito_chunks.jsonl", "r") as f:
    for line in f:
        all_chunks.append(json.loads(line))

np.random.seed(42)
cap = min(50, len(all_chunks))
sample_indices = np.random.choice(len(all_chunks), size=cap, replace=False)
sampled_chunks = [all_chunks[i] for i in sample_indices]

gold_records = []
for i, chunk in enumerate(sampled_chunks):
    window_idx = chunk.get('chunk_id', f'win_{i}') 
    
    gold_records.append({
        'item_id': chunk.get('chunk_id', f'chunk_{i}'),
        'cohort_era': window_idx,       
        'ts_start': chunk.get('ts_start', 'Check raw CSV'), # Added timestamp
        'ts_end': chunk.get('ts_end', 'Check raw CSV'),     # Added timestamp
        'text': chunk.get('text', ''),  
        'gold_trope': '',               
        'source': 'gold',               
        'annotator_id': 'YOUR_ID',      
        'annotation_sec': 0             
    })

df_gold_template = pd.DataFrame(gold_records)
df_gold_template.to_csv('gold_raw_template.csv', index=False)
print(f"Exported {len(df_gold_template)} chunks to gold_raw_template.csv for manual annotation.")

Exported 50 chunks to gold_raw_template.csv for manual annotation.


In [6]:
import json
import pandas as pd

# 1. Load your working gold CSV
# (Change the filename if you saved your manual annotations under a new name)
df_gold = pd.read_csv('gold_raw_template.csv') 

# 2. Extract the silver labels from the JSONL file
silver_labels = {}
with open("../silver_vocadito_chunks.jsonl", "r") as f:
    for line in f:
        chunk = json.loads(line)
        chunk_id = chunk.get('chunk_id')
        
        # Extract the raw silver trope, stripping the 'contour.' prefix
        trope_list = chunk.get('trope_ids', [])
        raw_trope = trope_list[0].replace('contour.', '') if trope_list else 'mixed'
        silver_labels[chunk_id] = raw_trope

# 3. Map the silver labels to your gold dataset based on item_id
df_gold['silver_trope'] = df_gold['item_id'].map(silver_labels)

# 4. Save the side-by-side comparison file
df_gold.to_csv('gold_vs_silver_comparison.csv', index=False)
print("Exported gold_vs_silver_comparison.csv")

# Optional: Run a quick agreement check if your gold_trope column is already filled out
completed_mask = df_gold['gold_trope'].notna() & (df_gold['gold_trope'] != '')
completed_df = df_gold[completed_mask]

if not completed_df.empty:
    agreement = (completed_df['gold_trope'] == completed_df['silver_trope']).mean()
    print(f"Current Silver-vs-Gold Agreement on completed rows: {agreement:.1%}")

Exported gold_vs_silver_comparison.csv


In [16]:
import pandas as pd
import re
import os

df_gold = pd.read_csv('gold_raw_template.csv')
search_root = "../" 
file_path_cache = {}

for root, dirs, files in os.walk(search_root):
    for file in files:
        if file.startswith("vocadito_") and file.endswith(".csv"):
            file_path_cache[file] = os.path.join(root, file)

def calculate_timestamps(row):
    item_id = str(row['item_id'])
    match = re.search(r"(vocadito_\d+):([A-Z0-9]+):w(\d+)", item_id)
    
    if not match:
        return pd.Series({'ts_start': None, 'ts_end': None})
        
    raw_prefix = match.group(1) 
    file_num = str(int(raw_prefix.split('_')[1])) 
    annotator = match.group(2)                    
    
    # FIX: The suffix is already the starting note index
    start_idx = int(match.group(3)) 
    end_idx = start_idx + 11
    
    expected_filename = f"vocadito_{file_num}_notes{annotator}.csv"
    
    if expected_filename not in file_path_cache:
        return pd.Series({'ts_start': None, 'ts_end': None})
        
    csv_path = file_path_cache[expected_filename]
    df_raw = pd.read_csv(csv_path, header=None, names=['onset', 'pitch', 'duration'])
    
    if start_idx >= len(df_raw):
        return pd.Series({'ts_start': None, 'ts_end': None})
        
    actual_end_idx = min(end_idx, len(df_raw) - 1)
    
    onset_sec = df_raw.iloc[start_idx]['onset']
    offset_sec = df_raw.iloc[actual_end_idx]['onset'] + df_raw.iloc[actual_end_idx]['duration']
    
    return pd.Series({'ts_start': onset_sec, 'ts_end': offset_sec})

df_gold[['ts_start', 'ts_end']] = df_gold.apply(calculate_timestamps, axis=1)

cols = ['item_id', 'cohort_era', 'ts_start', 'ts_end', 'text', 'gold_trope', 'source', 'annotator_id', 'annotation_sec']
df_gold = df_gold[[c for c in cols if c in df_gold.columns]]

df_gold.to_csv('gold_raw_template.csv', index=False)
print("Timestamps applied successfully. You can now verify the audio bounds.")

Timestamps applied successfully. You can now verify the audio bounds.


In [14]:
import pandas as pd
import re
import os

df_gold = pd.read_csv('gold_raw_template.csv')
search_root = "../" # Points to parent directory

row = df_gold.iloc[0]
item_id = str(row['item_id'])
print(f"--- DEBUGGING ITEM: {item_id} ---")

match = re.search(r"(vocadito_\d+):([A-Z0-9]+):w(\d+)", item_id)
if not match:
    print("1. Regex failed to parse item_id.")
else:
    raw_prefix = match.group(1) 
    file_num = str(int(raw_prefix.split('_')[1])) 
    annotator = match.group(2)                    
    window_idx = int(match.group(3))
    
    expected_filename = f"vocadito_{file_num}_notes{annotator}.csv"
    print(f"2. Looking for raw file named exactly: '{expected_filename}'")
    
    found_path = None
    for root, dirs, files in os.walk(search_root):
        if expected_filename in files:
            found_path = os.path.join(root, expected_filename)
            break
            
    if not found_path:
        print(f"3. ❌ FILE NOT FOUND in any subfolder of {search_root}")
        print("   Check if the raw files use zero-padding (e.g., 'vocadito_039_notesA1.csv') or a different extension.")
    else:
        print(f"3. ✅ File found at: {found_path}")
        df_raw = pd.read_csv(found_path)
        print(f"4. Columns in CSV: {list(df_raw.columns)}")
        print(f"5. Total rows in CSV: {len(df_raw)}")
        
        start_idx = window_idx * 6
        print(f"6. Calculated start note index: {start_idx}")
        
        if start_idx >= len(df_raw):
            print("   ❌ ERROR: Start index is beyond the end of the file!")
        else:
            onset_col = next((col for col in ['onset', 'onset_s'] if col in df_raw.columns), None)
            if onset_col:
                print(f"7. ✅ SUCCESS! Start timestamp: {df_raw.iloc[start_idx][onset_col]}")
            else:
                print("   ❌ ERROR: Could not find 'onset' or 'onset_s' column.")

--- DEBUGGING ITEM: vocadito_39:A1:w0000 ---
2. Looking for raw file named exactly: 'vocadito_39_notesA1.csv'
3. ✅ File found at: ../vocadito/Annotations/Notes/vocadito_39_notesA1.csv
4. Columns in CSV: ['1.259682540', '331.46', '0.208979592']
5. Total rows in CSV: 91
6. Calculated start note index: 0
   ❌ ERROR: Could not find 'onset' or 'onset_s' column.


In [3]:
import pandas as pd

# --- USER INPUT ---
ANNOTATOR_NAME = "Barak Snir"
GOLD_CSV_PATH  = "gold_raw.csv"  # Rename gold_raw_template.csv -> gold_raw.csv
# ------------------

VALID_LABELS = {
    "valley", "undulating", "arch", "descending",
    "ascending", "mixed", "static", "undetermined"
}

df_gold = pd.read_csv(GOLD_CSV_PATH)

# 1. Local CI/CD Gate Validation (mirrors validate_annotation_submission)
assert len(df_gold) <= 50, f"FAIL: Row count is {len(df_gold)} (> 50 scores 0!)."
assert (df_gold["source"] == "gold").all(), "FAIL: All rows must have source='gold'."
assert df_gold["gold_trope"].notna().all(), "FAIL: Missing labels in gold_trope."

df_gold["gold_trope"] = df_gold["gold_trope"].astype(str).str.strip().str.lower()
invalid = set(df_gold["gold_trope"]) - VALID_LABELS
assert not invalid, f"FAIL: Invalid labels found: {invalid}"

# 2. Time Validation Directly from `annotation_sec`
assert "annotation_sec" in df_gold.columns, "FAIL: Missing 'annotation_sec' column."
df_gold["annotation_sec"] = pd.to_numeric(df_gold["annotation_sec"], errors="raise")
assert df_gold["annotation_sec"].notna().all(), "FAIL: Null values in annotation_sec."
assert (df_gold["annotation_sec"] > 0).all(), "FAIL: All annotation_sec entries must be > 0."

total_sec  = float(df_gold["annotation_sec"].sum())
total_min  = total_sec / 60.0
total_hrs  = total_sec / 3600.0
mean_sec   = float(df_gold["annotation_sec"].mean())
median_sec = float(df_gold["annotation_sec"].median())
max_sec    = float(df_gold["annotation_sec"].max())

assert total_sec <= 7200.0, f"FAIL: Total effort {total_sec:.0f}s ({total_hrs:.2f}h) exceeds 7,200s (2.0h) cap!"

# Save cleaned gold_raw.csv
df_gold.to_csv(GOLD_CSV_PATH, index=False)

# 3. Write attestation.md
attestation_text = f"""# MP 9 Human Annotation Attestation

* **Annotator:** {ANNOTATOR_NAME}
* **Track / Pathway:** VDB Track — Pathway C / Audio-Only Human Verification (`vocadito`, Zenodo 5578807 v3)
* **Raw Gold File:** `{GOLD_CSV_PATH}`
* **Total Annotated Objects:** {len(df_gold)} (Gate: <= 50 objects — **PASS**)
* **Time Validation Source:** Cumulative per-item `annotation_sec` logged in `{GOLD_CSV_PATH}`
* **Total Logged Effort:** {total_sec:.0f} seconds ({total_min:.1f} minutes / {total_hrs:.2f} hours; Gate: <= 7,200s / 2.0 hours — **PASS**)
* **Per-Item Effort Summary:** Mean = {mean_sec:.1f}s | Median = {median_sec:.1f}s | Max = {max_sec:.0f}s

## Integrity Statement
I attest that all {len(df_gold)} rows in `{GOLD_CSV_PATH}` (`source=gold`) were labeled by me using the audio-only protocol defined in `codebook.md` without viewing MIDI sequences or using AI-generated labels, and that the cumulative human annotation time recorded in `annotation_sec` ({total_sec:.0f}s / {total_hrs:.2f} hours) is within the 2.0-hour cap.
"""

with open("attestation.md", "w", encoding="utf-8") as f:
    f.write(attestation_text)

print(f"PASS: Validated {len(df_gold)} gold rows.")
print(f"Total Effort (from annotation_sec): {total_sec:.0f}s ({total_min:.1f} min / {total_hrs:.2f} hrs) <= 2.0h cap.")
print("Generated attestation.md successfully.")


PASS: Validated 50 gold rows.
Total Effort (from annotation_sec): 1830s (30.5 min / 0.51 hrs) <= 2.0h cap.
Generated attestation.md successfully.


In [4]:
import pandas as pd

df_gold = pd.read_csv("gold_raw.csv")

# Sample 20 items deterministically for the overlap
overlap_df = df_gold.sample(n=20, random_state=42).copy()

# 1. Write overlap_ids.txt (required deliverable)
with open("overlap_ids.txt", "w") as f:
    for item_id in overlap_df["item_id"]:
        f.write(f"{item_id}\n")

# 2. Shuffle row order and strip gold_trope for a blind second pass
blind_df = overlap_df[["item_id", "ts_start", "ts_end"]].sample(frac=1, random_state=99).copy()
blind_df["rater2_trope"] = ""
blind_df.to_csv("overlap_blind_rater2.csv", index=False)

print("Wrote overlap_ids.txt (20 IDs) and overlap_blind_rater2.csv (shuffled, blind).")

Wrote overlap_ids.txt (20 IDs) and overlap_blind_rater2.csv (shuffled, blind).


In [6]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

df_gold = pd.read_csv("gold_raw.csv")
df_partner = pd.read_csv("partner_overlap_labeled.csv")
df_partner["partner_trope"] = df_partner["partner_trope"].astype(str).str.strip().str.lower()

# Merge on the 20 overlap IDs
merged = pd.merge(
    df_gold[["item_id", "gold_trope"]],
    df_partner[["item_id", "partner_trope"]],
    on="item_id",
    how="inner"
)
assert len(merged) == 20, f"Expected 20 overlap items, got {len(merged)}"

y1 = merged["gold_trope"].values
y2 = merged["partner_trope"].values

# 1. Point Estimates
pct_agree = float(np.mean(y1 == y2))
kappa = float(cohen_kappa_score(y1, y2))

# 2. 95% Percentile Bootstrap CI (B = 1,000)
rng = np.random.default_rng(42)
boot_kappas = []
for _ in range(1000):
    idx = rng.integers(0, len(y1), size=len(y1))
    k_val = cohen_kappa_score(y1[idx], y2[idx])
    if not np.isnan(k_val):
        boot_kappas.append(k_val)

ci_low, ci_high = np.percentile(boot_kappas, [2.5, 97.5])

# 3. Confusion Matrix
present_labels = sorted(list(set(y1) | set(y2)))
cm = pd.DataFrame(
    confusion_matrix(y1, y2, labels=present_labels),
    index=[f"A1:{l}" for l in present_labels],
    columns=[f"A2:{l}" for l in present_labels]
)

print(f"Percent Agreement : {pct_agree:.1%} ({int(np.sum(y1 == y2))}/20)")
print(f"Cohen's kappa     : {kappa:.3f} (95% Bootstrap CI: [{ci_low:.3f}, {ci_high:.3f}])\n")
print("Confusion Matrix (Rows = You, Cols = Partner):")
print(cm)

# 4. List Disagreements for Adjudication
disagreements = merged[merged["gold_trope"] != merged["partner_trope"]]
print(f"\nDisagreements to adjudicate ({len(disagreements)} items):")
print(disagreements.to_string(index=False))

# 5. Apply manual adjudications here (update dictionary after reviewing disagreements)
adjudications = {
    # Example: "vocadito_3:A2:w0030": "undulating",
}

df_adj = df_gold.copy()
df_adj["adjudicated_trope"] = df_adj.apply(
    lambda r: adjudications.get(r["item_id"], r["gold_trope"]), axis=1
)
df_adj["source"] = "gold"

df_adj.to_json("gold_adjudicated.jsonl", orient="records", lines=True)
print(f"\nExported {len(df_adj)} adjudicated rows to gold_adjudicated.jsonl.")

Percent Agreement : 100.0% (20/20)
Cohen's kappa     : 1.000 (95% Bootstrap CI: [1.000, 1.000])

Confusion Matrix (Rows = You, Cols = Partner):
               A2:arch  A2:ascending  A2:descending  A2:mixed  A2:undulating  \
A1:arch              4             0              0         0              0   
A1:ascending         0             4              0         0              0   
A1:descending        0             0              2         0              0   
A1:mixed             0             0              0         2              0   
A1:undulating        0             0              0         0              4   
A1:valley            0             0              0         0              0   

               A2:valley  
A1:arch                0  
A1:ascending           0  
A1:descending          0  
A1:mixed               0  
A1:undulating          0  
A1:valley              4  

Disagreements to adjudicate (0 items):
Empty DataFrame
Columns: [item_id, gold_trope, partner_trope]
Inde

In [7]:
import os
import re
import json
import math
import numpy as np
import pandas as pd
from collections import defaultdict

SEARCH_ROOT = "../"
WINDOW_SIZE = 12
STRIDE = 3

# 1. Load gold_raw.csv to strictly exclude gold windows (zero leakage)
df_gold = pd.read_csv("gold_raw.csv")
gold_item_ids = set(df_gold["item_id"].astype(str))

# 2. Helper: Convert Hz to nearest integer MIDI pitch
def hz_to_midi(hz: float) -> int:
    if hz <= 0 or np.isnan(hz):
        return 0
    return int(round(69 + 12 * math.log2(hz / 440.0)))

# 3. Weak-Supervision Labeling Functions (LFs)
def lf_geometric(notes: list[int]) -> str:
    """LF 1 (weight 0.50): Deterministic geometric contour heuristic."""
    if len(notes) < 3:
        return "undetermined"
    p_min, p_max = min(notes), max(notes)
    if (p_max - p_min) <= 2:
        return "static"
    
    # Count direction reversals with >= 2 semitone swing
    reversals = 0
    last_dir = 0
    anchor = notes[0]
    for p in notes[1:]:
        diff = p - anchor
        if abs(diff) >= 2:
            cur_dir = 1 if diff > 0 else -1
            if last_dir != 0 and cur_dir != last_dir:
                reversals += 1
            last_dir = cur_dir
            anchor = p
    if reversals >= 4:
        return "undulating"

    ends_max = max(notes[0], notes[-1])
    ends_min = min(notes[0], notes[-1])
    if (p_max - ends_max) >= 3 and (p_max - ends_max) >= (ends_min - p_min):
        return "arch"
    if (ends_min - p_min) >= 3:
        return "valley"
    
    net = notes[-1] - notes[0]
    if net >= 3:
        return "ascending"
    if net <= -3:
        return "descending"
    return "mixed"

def lf_linear_trend(notes: list[int]) -> str:
    """LF 2 (weight 0.30): OLS linear slope & range trend (abstains on non-monotonic)."""
    if len(notes) < 3:
        return "ABSTAIN"
    x = np.arange(len(notes), dtype=float)
    y = np.array(notes, dtype=float)
    slope = np.polyfit(x, y, 1)[0]
    total_range = float(np.max(y) - np.min(y))
    
    if total_range <= 2.0:
        return "static"
    if slope >= 0.35:
        return "ascending"
    if slope <= -0.35:
        return "descending"
    return "ABSTAIN"

def lf_midpoint_shape(notes: list[int]) -> str:
    """LF 3 (weight 0.20): Compares middle third mean against start/end thirds."""
    if len(notes) < 6:
        return "ABSTAIN"
    third = len(notes) // 3
    m_start = np.mean(notes[:third])
    m_mid   = np.mean(notes[third:2*third])
    m_end   = np.mean(notes[2*third:])
    
    if (m_mid - max(m_start, m_end)) >= 2.0:
        return "arch"
    if (min(m_start, m_end) - m_mid) >= 2.0:
        return "valley"
    return "ABSTAIN"

def aggregate_weighted_votes(notes: list[int]) -> tuple[str, dict]:
    votes = defaultdict(float)
    l1 = lf_geometric(notes)
    l2 = lf_linear_trend(notes)
    l3 = lf_midpoint_shape(notes)
    
    for lbl, w in [(l1, 0.50), (l2, 0.30), (l3, 0.20)]:
        if lbl != "ABSTAIN":
            votes[lbl] += w
            
    best_label = max(votes.items(), key=lambda kv: kv[1])[0]
    return best_label, {"lf1_geo": l1, "lf2_ols": l2, "lf3_mid": l3}

# 4. Walk directory, slice with stride=3, exclude gold IDs, and label silver pool
silver_records = []
file_pattern = re.compile(r"vocadito_(\d+)_notes(A[12])\.csv$")

for root, _, files in os.walk(SEARCH_ROOT):
    for fname in sorted(files):
        m = file_pattern.match(fname)
        if not m:
            continue
        clip_num = int(m.group(1))
        annotator = m.group(2)
        fpath = os.path.join(root, fname)
        
        df_raw = pd.read_csv(fpath, header=None, names=["onset", "pitch_hz", "duration"])
        midi_seq = [hz_to_midi(h) for h in df_raw["pitch_hz"].values]
        
        n_notes = len(midi_seq)
        for start_idx in range(0, max(1, n_notes - WINDOW_SIZE + 1), STRIDE):
            end_idx = min(start_idx + WINDOW_SIZE, n_notes)
            window_notes = midi_seq[start_idx:end_idx]
            if len(window_notes) < 3:
                continue
                
            chunk_id = f"vocadito_{clip_num}:{annotator}:w{start_idx:04d}"
            
            # Strict gold-set exclusion to prevent evaluation leakage
            if chunk_id in gold_item_ids:
                continue
                
            ts_start = float(df_raw.iloc[start_idx]["onset"])
            ts_end = float(df_raw.iloc[end_idx - 1]["onset"] + df_raw.iloc[end_idx - 1]["duration"])
            
            silver_label, lf_outputs = aggregate_weighted_votes(window_notes)
            
            silver_records.append({
                "chunk_id": chunk_id,
                "clip_id": f"vocadito_{clip_num}",
                "annotator": annotator,
                "cohort_window_idx": start_idx // STRIDE,  # Ordered cohort axis
                "ts_start": round(ts_start, 4),
                "ts_end": round(ts_end, 4),
                "text": " ".join(str(n) for n in window_notes),
                "silver_trope": silver_label,
                "lf_votes": lf_outputs,
                "source": "silver"
            })

assert len(silver_records) >= 1000, f"Silver pool has only {len(silver_records)} chunks (< 1,000)!"

with open("silver_pool_expanded.jsonl", "w", encoding="utf-8") as f:
    for rec in silver_records:
        f.write(json.dumps(rec) + "\n")

print(f"SUCCESS: Generated {len(silver_records)} leak-free silver chunks (source='silver').")
print("Silver label distribution:")
print(pd.Series([r["silver_trope"] for r in silver_records]).value_counts())

SUCCESS: Generated 1076 leak-free silver chunks (source='silver').
Silver label distribution:
valley        297
undulating    286
arch          274
descending    109
ascending      78
mixed          28
static          4
Name: count, dtype: int64


In [8]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

LABEL_SPACE = [
    "valley", "undulating", "arch", "descending",
    "ascending", "mixed", "static", "undetermined"
]

# 1. Load adjudicated gold set
gold_records = [json.loads(line) for line in open("gold_adjudicated.jsonl")]
df_gold = pd.DataFrame(gold_records)
assert len(df_gold) <= 50, "CI GATE VIOLATION: gold set exceeds 50 rows!"
assert (df_gold["source"] == "gold").all(), "INV-17 VIOLATION: non-gold row found!"

# 2. Load expanded silver pool (verify >= 1,000 chunks and source=silver)
silver_records = [json.loads(line) for line in open("silver_pool_expanded.jsonl")]
df_silver = pd.DataFrame(silver_records)
assert len(df_silver) >= 1000, f"Silver pool has {len(df_silver)} rows; MP 9 requires >= 1,000."
assert (df_silver["source"] == "silver").all(), "All silver rows must carry source='silver'."

# Helper to extract label consistently across schemas
def get_silver_label(rec):
    if "silver_trope" in rec and pd.notna(rec["silver_trope"]):
        return str(rec["silver_trope"]).replace("contour.", "")
    if "trope_ids" in rec and rec["trope_ids"]:
        return str(rec["trope_ids"][0]).replace("contour.", "")
    return "mixed"

# 3. Also load original silver labels for the 50 gold IDs to measure silver-vs-gold agreement
orig_silver_map = {}
for path in ["../silver_vocadito_chunks.jsonl", "silver_pool_expanded.jsonl"]:
    try:
        with open(path) as f:
            for line in f:
                r = json.loads(line)
                orig_silver_map[r["chunk_id"]] = get_silver_label(r)
    except FileNotFoundError:
        pass

df_gold["silver_pred"] = df_gold["item_id"].map(orig_silver_map).fillna("mixed")

y_true = df_gold["gold_trope"].values
y_silv = df_gold["silver_pred"].values

ws_acc = accuracy_score(y_true, y_silv)
ws_f1  = f1_score(y_true, y_silv, average="macro", zero_division=0)

# Bootstrap 95% CI for silver-vs-gold Macro-F1
rng = np.random.default_rng(42)
boot_ws_f1 = [
    f1_score(y_true[idx], y_silv[idx], average="macro", zero_division=0)
    for idx in (rng.integers(0, len(y_true), size=len(y_true)) for _ in range(1000))
]
ws_ci = [float(np.percentile(boot_ws_f1, 2.5)), float(np.percentile(boot_ws_f1, 97.5))]

print(f"Silver Pool Size : {len(df_silver)} chunks (source='silver')")
print(f"Silver-vs-Gold   : Accuracy = {ws_acc:.3f} | Macro-F1 = {ws_f1:.3f} (95% CI: [{ws_ci[0]:.3f}, {ws_ci[1]:.3f}])")

# Inspect top silver failure modes for report.md
disagreements = df_gold[df_gold["gold_trope"] != df_gold["silver_pred"]]
print("\nTop Silver Failure Modes (Gold -> Silver):")
print(disagreements.groupby(["gold_trope", "silver_pred"]).size().sort_values(ascending=False).head(10))

Silver Pool Size : 1076 chunks (source='silver')
Silver-vs-Gold   : Accuracy = 0.620 | Macro-F1 = 0.591 (95% CI: [0.416, 0.726])

Top Silver Failure Modes (Gold -> Silver):
gold_trope  silver_pred
undulating  valley         4
ascending   undulating     3
            arch           3
undulating  descending     2
mixed       valley         2
undulating  arch           2
arch        undulating     1
mixed       descending     1
valley      undulating     1
dtype: int64


In [9]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Enforce AMD-to-CPU fallback before PyTorch

import re
import faiss
import torch
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

def compute_embedding(notes: list[int]) -> np.ndarray:
    vec = np.zeros(37, dtype=np.float32)
    if not notes:
        return vec
    for note in notes:
        vec[note % 12] += 1.0
    for i in range(1, len(notes)):
        interval = max(-12, min(12, notes[i] - notes[i - 1]))
        vec[12 + interval + 12] += 1.0
    return vec

# 1. Strictly exclude all gold item_ids from the retrieval index (zero leakage)
gold_ids = set(df_gold["item_id"])
index_records = [r for r in silver_records if r["chunk_id"] not in gold_ids]
assert len(index_records) >= 1000, f"Leak-free index has {len(index_records)} chunks (< 1,000)."

silver_vecs = np.vstack([
    compute_embedding([int(n) for n in r["text"].split()]) for r in index_records
]).astype(np.float32)
faiss.normalize_L2(silver_vecs)

base_index = faiss.IndexFlatIP(37)
faiss_index = faiss.IndexIDMap(base_index)
int_ids = np.arange(len(index_records), dtype=np.int64)
faiss_index.add_with_ids(silver_vecs, int_ids)

silver_labels_arr = [get_silver_label(r) for r in index_records]
silver_texts_arr  = [r["text"] for r in index_records]

# Precompute total relevant silver items per class for Recall@k
total_rel_by_class = pd.Series(silver_labels_arr).value_counts().to_dict()

# 2. Initialize Qwen/Qwen1.5-1.8B-Chat on CPU (50 gold queries * 3 k-values = 150 inferences, ~3 min)
model_id = "Qwen/Qwen1.5-1.8B-Chat"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)
llm_pipe = pipeline(
    "text-generation", model=model, tokenizer=tokenizer,
    max_new_tokens=3, temperature=0.0, do_sample=False, return_full_text=False
)

SYS_PROMPT = (
    "You are an expert music analyst classifying 12-note MIDI pitch contours into exactly one of: "
    "valley, undulating, arch, descending, ascending, mixed, static, undetermined. "
    "Output only the exact lowercase label."
)

def rag_llm_predict(query_text: str, retrieved_pairs: list[tuple[str, str]]) -> str:
    messages = [{"role": "system", "content": SYS_PROMPT}]
    for ex_text, ex_lbl in retrieved_pairs:
        messages.append({"role": "user", "content": f"Sequence: {ex_text}"})
        messages.append({"role": "assistant", "content": ex_lbl})
    messages.append({"role": "user", "content": f"Sequence: {query_text}"})
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = llm_pipe(prompt)[0]["generated_text"].strip().lower()
    clean = re.sub(r"[^a-z]", "", out)
    return clean if clean in LABEL_SPACE else "mixed"

# 3. Evaluate Retrieval (P@k, Recall@k) + RAG kNN + RAG LLM across k in {3, 5, 10}
rag_results = {}
for k in [3, 5, 10]:
    p_at_k_list, r_at_k_list = [], []
    knn_preds, llm_preds = [], []

    for _, row in df_gold.iterrows():
        q_notes = [int(n) for n in str(row["text"]).split()]
        q_vec = compute_embedding(q_notes).reshape(1, -1)
        faiss.normalize_L2(q_vec)

        scores, idxs = faiss_index.search(q_vec, k)
        ret_idxs = [int(i) for i in idxs[0] if i != -1]
        ret_scores = [float(s) for s, i in zip(scores[0], idxs[0]) if i != -1]
        ret_labels = [silver_labels_arr[i] for i in ret_idxs]
        ret_pairs  = [(silver_texts_arr[i], silver_labels_arr[i]) for i in ret_idxs]

        # Retrieval relevance: silver_label == gold_trope
        gold_lbl = row["gold_trope"]
        hits = sum(1 for lbl in ret_labels if lbl == gold_lbl)
        p_at_k_list.append(hits / k)
        denom = max(1, min(k, total_rel_by_class.get(gold_lbl, 0)))
        r_at_k_list.append(hits / denom)

        # Similarity-weighted kNN prediction
        vote = defaultdict(float)
        for lbl, sc in zip(ret_labels, ret_scores):
            vote[lbl] += max(sc, 0.0)
        knn_preds.append(max(vote, key=vote.get) if vote else "mixed")

        # RAG-conditioned LLM prediction
        llm_preds.append(rag_llm_predict(row["text"], ret_pairs))

    # 95% Percentile Bootstrap CIs (B = 1,000)
    rng = np.random.default_rng(42)
    n = len(df_gold)
    boot_p, boot_r, boot_knn, boot_llm = [], [], [], []
    y_gold = df_gold["gold_trope"].values
    knn_arr, llm_arr = np.array(knn_preds), np.array(llm_preds)
    p_arr, r_arr = np.array(p_at_k_list), np.array(r_at_k_list)

    for _ in range(1000):
        idx = rng.integers(0, n, size=n)
        boot_p.append(np.mean(p_arr[idx]))
        boot_r.append(np.mean(r_arr[idx]))
        boot_knn.append(f1_score(y_gold[idx], knn_arr[idx], average="macro", zero_division=0))
        boot_llm.append(f1_score(y_gold[idx], llm_arr[idx], average="macro", zero_division=0))

    def ci_dict(val, arr):
        return {
            "point": float(val),
            "ci_95": [float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))]
        }

    rag_results[f"k={k}"] = {
        "P@k": ci_dict(np.mean(p_arr), boot_p),
        "Recall@k": ci_dict(np.mean(r_arr), boot_r),
        "RAG_kNN_MacroF1": ci_dict(f1_score(y_gold, knn_arr, average="macro", zero_division=0), boot_knn),
        "RAG_LLM_MacroF1": ci_dict(f1_score(y_gold, llm_arr, average="macro", zero_division=0), boot_llm),
    }
    print(f"k={k:2d} | P@k={rag_results[f'k={k}']['P@k']['point']:.3f} | "
          f"Recall@k={rag_results[f'k={k}']['Recall@k']['point']:.3f} | "
          f"RAG-kNN F1={rag_results[f'k={k}']['RAG_kNN_MacroF1']['point']:.3f} | "
          f"RAG-LLM F1={rag_results[f'k={k}']['RAG_LLM_MacroF1']['point']:.3f}")

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/vscode/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:410: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packag

k= 3 | P@k=0.387 | Recall@k=0.387 | RAG-kNN F1=0.368 | RAG-LLM F1=0.086


/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:410: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:415: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


k= 5 | P@k=0.336 | Recall@k=0.336 | RAG-kNN F1=0.382 | RAG-LLM F1=0.043


/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:410: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:415: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


k=10 | P@k=0.284 | Recall@k=0.284 | RAG-kNN F1=0.270 | RAG-LLM F1=0.101


In [13]:
import torch.nn as nn

# 1. Parse ordered cohort axis (event-time window index binned into 10 sequential bins 0..9)
def extract_cohort_bin(chunk_id: str, bin_width: int = 6, max_bin: int = 9) -> int:
    m = re.search(r":w(\d+)$", str(chunk_id))
    start_note = int(m.group(1)) if m else 0
    return min(start_note // bin_width, max_bin)

df_conv = pd.DataFrame({
    "chunk_id": [r["chunk_id"] for r in silver_records],
    "cohort":   [extract_cohort_bin(r["chunk_id"]) for r in silver_records],
    "label":    [get_silver_label(r) for r in silver_records],
})

cohorts = sorted(df_conv["cohort"].unique())
T = len(cohorts)

# Build per-cohort frequency matrix M of shape (len(LABEL_SPACE), T)
freq_matrix = np.zeros((len(LABEL_SPACE), T), dtype=np.float32)
for t_idx, c in enumerate(cohorts):
    sub = df_conv[df_conv["cohort"] == c]
    counts = sub["label"].value_counts(normalize=True)
    for l_idx, lbl in enumerate(LABEL_SPACE):
        freq_matrix[l_idx, t_idx] = counts.get(lbl, 0.0)

# 2. Fixed Kernels + Stratified Cohort Bootstrap (B = 1,000) for top contour class
target_contour = df_conv["label"].value_counts().idxmax()
k_smooth = np.array([0.25, 0.50, 0.25])
# Note: np.convolve flips the kernel, so [1, -1] computes v[t+1] - v[t] (forward drift [-1, 1])
k_diff   = np.array([1.0, -1.0])

rng = np.random.default_rng(42)
cohort_groups = {c: df_conv[df_conv["cohort"] == c]["label"].values for c in cohorts}
boot_smooth = np.zeros((1000, T))
boot_diff   = np.zeros((1000, T - 1))

for b in range(1000):
    rates = np.array([
        np.mean(rng.choice(cohort_groups[c], size=len(cohort_groups[c]), replace=True) == target_contour)
        for c in cohorts
    ])
    boot_smooth[b] = np.convolve(np.pad(rates, (1, 1), mode="edge"), k_smooth, mode="valid")
    boot_diff[b]   = np.convolve(rates, k_diff, mode="valid")

print(f"=== Cohort Convolution Analysis (Target Contour: '{target_contour}') ===")
for i, c in enumerate(cohorts):
    sm_m, sm_lo, sm_hi = np.mean(boot_smooth[:, i]), np.percentile(boot_smooth[:, i], 2.5), np.percentile(boot_smooth[:, i], 97.5)
    print(f"Cohort {c}: Smoothed={sm_m:.3f} (95% CI [{sm_lo:.3f}, {sm_hi:.3f}])")

print("\nFirst-Difference Drift (Delta_t = Cohort_{t+1} - Cohort_t):")
for i in range(T - 1):
    dm, dlo, dhi = np.mean(boot_diff[:, i]), np.percentile(boot_diff[:, i], 2.5), np.percentile(boot_diff[:, i], 97.5)
    print(f"Cohort {cohorts[i]}->{cohorts[i+1]}: Drift={dm:+.3f} (95% CI [{dlo:+.3f}, {dhi:+.3f}])")

import torch
import torch.nn as nn

# 3. CIS 731 Requirement: Learned 1-D PyTorch Conv Kernel vs. Fixed Kernel Forecast Error
# Causal 1-step-ahead forecast: predict cohort t from [t-3, t-2, t-1] across all 8 contour classes
torch.manual_seed(42)
X_all = torch.tensor(freq_matrix, dtype=torch.float32).unsqueeze(1)  # Shape: (8, 1, T=10)

split_t = T - 2  # Train targets: cohorts 3..7 (5 steps); Held-out test targets: cohorts 8..9 (2 steps)
X_train = X_all[:, :, :split_t - 1]      # Indices 0..6 (len 7) -> conv1d outputs 5 predictions
y_train = X_all[:, :, 3:split_t]         # Indices 3..7 (len 5) -> zero leakage into test cohorts
X_test  = X_all[:, :, split_t - 3:T - 1] # Indices 5..8 (len 4) -> conv1d outputs 2 predictions
y_test  = X_all[:, :, split_t:T]         # Indices 8..9 (len 2)

conv1d = nn.Conv1d(in_channels=1, out_channels=1, kernel_size=3, bias=False)
optimizer = torch.optim.Adam(conv1d.parameters(), lr=0.05)
loss_fn = nn.MSELoss()

for epoch in range(300):
    optimizer.zero_grad()
    preds = conv1d(X_train)
    loss = loss_fn(preds, y_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    learned_preds = conv1d(X_test)
    learned_mse = float(loss_fn(learned_preds, y_test).item())
    learned_mae = float(torch.mean(torch.abs(learned_preds - y_test)).item())

    # Fixed [0.25, 0.5, 0.25] causal forecast on the exact same held-out windows
    fixed_w = torch.tensor([[[0.25, 0.50, 0.25]]], dtype=torch.float32)
    fixed_preds = nn.functional.conv1d(X_test, fixed_w)
    fixed_mse = float(loss_fn(fixed_preds, y_test).item())
    fixed_mae = float(torch.mean(torch.abs(fixed_preds - y_test)).item())

learned_weights = conv1d.weight.detach().flatten().tolist()
print("=== CIS 731 Learned 1-D PyTorch Conv vs. Fixed Kernel (Held-Out Forecast) ===")
print(f"Learned Kernel Weights : {[round(w, 4) for w in learned_weights]}")
print(f"Learned 1-D Conv       : Held-out MSE = {learned_mse:.6f} | MAE = {learned_mae:.4f}")
print(f"Fixed [0.25, 0.5, 0.25]: Held-out MSE = {fixed_mse:.6f} | MAE = {fixed_mae:.4f}")

=== Cohort Convolution Analysis (Target Contour: 'valley') ===
Cohort 0: Smoothed=0.263 (95% CI [0.212, 0.316])
Cohort 1: Smoothed=0.293 (95% CI [0.252, 0.334])
Cohort 2: Smoothed=0.286 (95% CI [0.241, 0.330])
Cohort 3: Smoothed=0.264 (95% CI [0.222, 0.309])
Cohort 4: Smoothed=0.283 (95% CI [0.236, 0.336])
Cohort 5: Smoothed=0.265 (95% CI [0.209, 0.325])
Cohort 6: Smoothed=0.221 (95% CI [0.171, 0.276])
Cohort 7: Smoothed=0.267 (95% CI [0.198, 0.334])
Cohort 8: Smoothed=0.328 (95% CI [0.243, 0.410])
Cohort 9: Smoothed=0.316 (95% CI [0.243, 0.395])

First-Difference Drift (Delta_t = Cohort_{t+1} - Cohort_t):
Cohort 0->1: Drift=+0.065 (95% CI [-0.027, +0.160])
Cohort 1->2: Drift=-0.010 (95% CI [-0.114, +0.094])
Cohort 2->3: Drift=-0.075 (95% CI [-0.175, +0.026])
Cohort 3->4: Drift=+0.075 (95% CI [-0.029, +0.178])
Cohort 4->5: Drift=-0.001 (95% CI [-0.132, +0.127])
Cohort 5->6: Drift=-0.143 (95% CI [-0.262, -0.029])
Cohort 6->7: Drift=+0.112 (95% CI [-0.026, +0.249])
Cohort 7->8: Drift=+0.

In [14]:
results_payload = {
    "gold_count": int(len(df_gold)),
    "silver_pool_count": int(len(df_silver)),
    "weak_supervision": {
        "accuracy": float(ws_acc),
        "macro_f1": float(ws_f1),
        "macro_f1_ci95": ws_ci
    },
    "mp7_baseline": {
        "model": "Qwen/Qwen1.5-1.8B-Chat (fixed demos)",
        "macro_f1": 0.016,
        "macro_f1_ci95": [0.008, 0.025]
    },
    "rag_classification": rag_results,
    "cohort_convolution": {
        "cohort_axis": "event_time_window_index_binned_0_to_9",
        "target_contour": target_contour,
        "cis731_learned_kernel_weights": learned_weights,
        "cis731_learned_mse": learned_mse,
        "cis731_learned_mae": learned_mae,
        "fixed_smoothing_mse": fixed_mse,
        "fixed_smoothing_mae": fixed_mae
    }
}

with open("results.json", "w") as f:
    json.dump(results_payload, f, indent=2)

print("Saved results.json successfully.")

Saved results.json successfully.


---

## Part 7: Problem Set 8 (PS 8) & MP 9 Deliverables Checklist

**Action Items Before Submitting PS 8 / MP 9:**
1. **Human Annotation Attestation (INV-17):** Verify that every item in `gold_standard_eval_v1.jsonl` was manually transcribed/labeled by a human team member (`source='gold'`), staying within the **<= 50 items / <= 2.0 hours** individual workload cap.
2. **Explicit Normalization Specification:** Document your exact regex, digit-expansion, and disfluency (`<hesitation>` vs. stripping) rules in a Markdown cell prior to reporting WER.
3. **Complete Error Breakdown:** Report exact S, D, I, C, N counts alongside corpus WER and Entity Error Rate (EER).
4. **95% Percentile Bootstrap CIs (B = 1000):** Include the [2.5th, 97.5th] percentile confidence intervals for your primary metric and paired model difference (delta_F1 or delta_WER).
